In [ ]:
import numpy.fft as fft
import numpy as np
from matplotlib import pyplot as plt
import os

In [ ]:

config = {
    'sdr_center_freq': 2402e6,
    'sdr_sample_rate': 2000000,
    'sdr_num_samples': 2**20,
    'sdr_meas_paths': ['../disk/chamber_test/open', '../disk/chamber_test/beacons', '../disk/chamber_test/background'],

    'burst_cnt' : 10, # How many bursts to use when generating the spectra.
    'filter' : 'max', # 'max' or 'avg' to use the maximum or the average of the above mentioned number of bursts.

    'packet_thresh': 1000, # Thershold to use for detecting packets.
    'packet_paths': ['../disk/chamber_test/beacons'] # Path to the measurements used to create the packet spectra.
}

In [ ]:
# Create the data output array and fill it with zeros
data_fft = [np.zeros(config['sdr_num_samples']) for meas_path in config['sdr_meas_paths']]

# Read and transform the data files.
for meas_path, index in zip(config['sdr_meas_paths'], range(0,config['sdr_num_samples'])):
    files = list(filter(lambda k: '.npy' in k, os.listdir(meas_path)))
    for file in files[:config['burst_cnt']]:
        data_iq = np.load(meas_path+'/'+file)
        data_transformed = np.abs(fft.fft(np.hanning(len(data_iq)) * data_iq, norm='ortho'))

        # Apply the filter as set by config['filter']
        if config['filter'] == 'max':
            data_fft[index] = np.maximum(data_transformed, data_fft[index])
        elif config['filter'] == 'avg':
            data_fft[index] += np.abs(data_transformed) / config['burst_cnt']
        else:
            raise ValueError('The value of config["filter"] must be either "max" or "avg".')

In [ ]:
packets_fft = []

for data_path in config['packet_paths']:
    files = list(filter(lambda k: '.npy' in k, os.listdir(data_path)))
    packets_iq = []
    for index, file in enumerate(files):
        data_iq = np.load(data_path+'/'+file)
        data_iq = data_iq[np.abs(data_iq) > config['packet_thresh']]
        packets_iq = np.concatenate((packets_iq, data_iq))
        if len(packets_iq) >= config['sdr_num_samples']:
            packets_iq = packets_iq[:config['sdr_num_samples']]
            break
    packets_fft.append(np.abs(fft.fft(np.hanning(len(packets_iq)) * packets_iq, config['sdr_num_samples'], norm='ortho')))

In [ ]:
# Frequency Domain Plots
# plt.switch_backend('TkAgg') # Use pop-up plots

freq = np.linspace(-config['sdr_sample_rate']/2, config['sdr_sample_rate']/2, config['sdr_num_samples'])

for meas_path, result in zip(config['packet_paths'], packets_fft):
    plt.plot(
        freq,
        np.fft.fftshift(np.clip(20*np.log10(result/2**12),-80,80)),
        label=meas_path.split('/')[-1]
    )

for meas_path, result in zip(config['sdr_meas_paths'], data_fft):
    plt.plot(
        freq,
        np.fft.fftshift(np.clip(20*np.log10(result/2**12),-80,80)),
        label=meas_path.split('/')[-1]
    )

plt.legend(loc='upper left')
plt.show()

In [ ]:
# Time Domain Plots
# plt.switch_backend('TkAgg') # Use pop-up plots

time = np.linspace(0, config['burst_cnt']*config['sdr_num_samples']/config['sdr_sample_rate'], config['burst_cnt']*config['sdr_num_samples'])

for meas_path, index in zip(config['sdr_meas_paths'], range(0,config['sdr_num_samples'])):
    files = list(sorted(filter(lambda k: '.npy' in k, os.listdir(meas_path))))
    iq_data = []
    for file in files[:config['burst_cnt']]:
        iq_data = np.concatenate((iq_data, np.load(meas_path+'/'+file)))
    plt.plot(
        time,
        np.abs(iq_data),
        label=meas_path.split('/')[-1]
    )

plt.legend(loc='upper left')
plt.show()